In [1]:
!pip install -q --upgrade \
accelerate==0.30.1 \
bitsandbytes \
sentencepiece \
safetensors \
fastapi \
uvicorn \
pyngrok
!pip install -U transformers tokenizers accelerate

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.30.1
    Uninstalling accelerate-0.30.1:
      Successfully uninstalled accelerate-0.30.1


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import torch
import re

# Prevent CUDA fragmentation for long-running servers
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ==============================
# DRIVE PATHS
# ==============================

DRIVE_ROOT = "/content/drive/MyDrive/Goober"

MODEL_DIR = os.path.join(DRIVE_ROOT, "Models")

MODEL_NAME = "smollm3-3b"

MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME)

print("Model path:", MODEL_PATH)

# ==============================
# GENERATION SETTINGS
# ==============================

MAX_NEW_TOKENS = 30
TEMPERATURE = 0.95
TOP_P = 0.85
TOP_K = 10
REPETITION_PENALTY = 1.3

# ==============================
# SENTENCE STREAMING
# ==============================

SENTENCE_REGEX = re.compile(r"[.!]")

# ==============================
# DEVICE
# ==============================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

# ==============================
# API SERVER
# ==============================

PORT = 8000

Model path: /content/drive/MyDrive/Goober/Models/smollm3-3b
Using device: cuda


In [4]:
import os
os.system("fuser -k 8000/tcp")
torch.backends.cuda.matmul.allow_tf32 = True

import time
import threading
import torch

from fastapi import FastAPI, Request
from fastapi.responses import PlainTextResponse

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn


# NGROK auth token
NGROK_AUTH_TOKEN = "3AvKB1I58T0mXO6fc1r1ScEGv4K_7N7d7DSJJeQCUVoP9FrZC"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


# =================================
# LOAD MODEL (4-BIT QUANTIZED)
# =================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

print("Configuring 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = True
model.eval()

print("Model loaded successfully")


# =================================
# GENERATION FUNCTION
# =================================

def generate_response(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            use_cache=True,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[-1]:]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return text.strip()


# =================================
# FASTAPI SERVER
# =================================

app = FastAPI()


@app.post("/generate")
async def generate(request: Request):

    data = await request.json()

    messages = data.get("messages", [])

    # APPLY CHAT TEMPLATE (important)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    response_text = generate_response(prompt)

    return PlainTextResponse(response_text)


# =================================
# START SERVER THREAD
# =================================

import asyncio
from uvicorn import Config, Server

async def serve():
    config = Config(
        app=app,
        host="0.0.0.0",
        port=PORT,
        log_level="warning"
    )
    server = Server(config)
    await server.serve()

def run_server():
    asyncio.run(serve())

print("Starting inference server thread...")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()


# =================================
# START NGROK
# =================================

time.sleep(2)

print("Opening ngrok tunnel...")

public_url = ngrok.connect(PORT).public_url

print("\n==============================")
print("API ENDPOINT:")
print(public_url + "/generate")
print("==============================\n")


# =================================
# KEEP NOTEBOOK ALIVE
# =================================

while True:
    time.sleep(3600)

Loading tokenizer...
Configuring 4-bit quantization...
Loading model...


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Model loaded successfully
Starting inference server thread...
Opening ngrok tunnel...

API ENDPOINT:
https://prosupervision-accusatively-reda.ngrok-free.dev/generate



KeyboardInterrupt: 